# Marcos de Canaveses — Ponte GPU T4 (Google Colab)

1. No menu superior: **Ambiente de execução → Alterar tipo de ambiente de execução → GPU T4**.
2. Execute as células em ordem.
3. Copie a URL `https://....trycloudflare.com` exibida na última célula e cole no chat.

In [ ]:
# Célula 1: Dependências e verificador de GPU
!nvidia-smi
!pip -q install --upgrade diffusers transformers accelerate peft safetensors fastapi uvicorn pillow imageio[ffmpeg]
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('Dependências instaladas!')

In [ ]:
# Célula 2: Carregar pipeline Pixel Art (SDXL + Pixel Art XL LoRA) otimizado para T4 15GB
import torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

assert torch.cuda.is_available(), 'GPU T4 não detectada! Ative a GPU nas configurações do Colab.'

MODEL_ID = 'stabilityai/stable-diffusion-xl-base-1.0'
LORA_ID = 'nerijs/pixel-art-xl'

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.load_lora_weights(LORA_ID)
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
pipe.enable_model_cpu_offload()
print('Pipeline Pixel Art SDXL otimizado para T4 carregado!')

In [ ]:
# Célula 3: Subir API FastAPI + Túnel Público Cloudflare (com diagnóstico de erro)
import base64, io, os, re, subprocess, threading, time, tempfile, traceback
import torch
from PIL import Image
from fastapi import FastAPI
from fastapi.responses import Response, JSONResponse
from pydantic import BaseModel
import uvicorn

app = FastAPI(title='Marcos de Canaveses T4 Bridge')

class ImageReq(BaseModel):
    prompt: str
    negative_prompt: str = 'blurry, 3d render, photorealistic, text, watermark, signature, lowres, messy'
    gen_width: int = 1024
    gen_height: int = 576
    out_width: int = 640
    out_height: int = 360
    steps: int = 24
    guidance: float = 7.0
    seed: int = 42
    lora_scale: float = 1.0

class VideoOgvReq(BaseModel):
    prompt: str
    negative_prompt: str = 'blurry, text, watermark, lowres'
    out_width: int = 640
    out_height: int = 360
    duration_sec: float = 4.0
    fps: int = 15
    seed: int = 42

@app.get('/health')
def health():
    return {'ok': True, 'gpu': torch.cuda.get_device_name(0)}

@app.post('/generate_image')
def generate_image(req: ImageReq):
    try:
        gen = torch.Generator('cpu').manual_seed(req.seed)
        full_prompt = f'pixel art, {req.prompt}, crisp pixels, retro rpg game art'
        kwargs = {
            'prompt': full_prompt,
            'negative_prompt': req.negative_prompt,
            'width': req.gen_width,
            'height': req.gen_height,
            'num_inference_steps': req.steps,
            'guidance_scale': req.guidance,
            'generator': gen,
        }
        if abs(req.lora_scale - 1.0) > 0.01:
            kwargs['cross_attention_kwargs'] = {'scale': req.lora_scale}
        img = pipe(**kwargs).images[0]
        if req.out_width > 0 and req.out_height > 0:
            img = img.resize((req.out_width, req.out_height), Image.Resampling.NEAREST)
        buf = io.BytesIO()
        img.save(buf, format='PNG')
        return {'png_base64': base64.b64encode(buf.getvalue()).decode('ascii')}
    except Exception:
        return JSONResponse(status_code=500, content={'error': traceback.format_exc()})

@app.post('/generate_kenburns_ogv')
def generate_kenburns_ogv(req: VideoOgvReq):
    try:
        gen = torch.Generator('cpu').manual_seed(req.seed)
        img = pipe(
            prompt=f'pixel art, {req.prompt}, atmospheric medieval portuguese scene',
            negative_prompt=req.negative_prompt,
            width=1024,
            height=576,
            num_inference_steps=24,
            guidance_scale=7.0,
            generator=gen,
        ).images[0]

        with tempfile.TemporaryDirectory() as tmp:
            src_png = os.path.join(tmp, 'base.png')
            out_ogv = os.path.join(tmp, 'out.ogv')
            img.save(src_png)
            total_frames = max(12, int(req.duration_sec * req.fps))
            vf = (
                f"zoompan=z='min(zoom+0.0018,1.18)':d={total_frames}:"
                f"x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':s={req.out_width}x{req.out_height}:fps={req.fps},"
                "format=yuv420p"
            )
            cmd = [
                'ffmpeg', '-y', '-loop', '1', '-i', src_png,
                '-vf', vf, '-t', str(req.duration_sec),
                '-pix_fmt', 'yuv420p', '-c:v', 'libtheora', '-q:v', '6', '-an', out_ogv
            ]
            res = subprocess.run(cmd, capture_output=True, text=True)
            if res.returncode != 0:
                return JSONResponse(status_code=500, content={'error': res.stderr})
            with open(out_ogv, 'rb') as f:
                data = f.read()
        return Response(content=data, media_type='video/ogg')
    except Exception:
        return JSONResponse(status_code=500, content={'error': traceback.format_exc()})

def run_server():
    uvicorn.run(app, host='127.0.0.1', port=8765, log_level='warning')

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8765'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
for line in proc.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('\n========================================')
        print('COLAB T4 BRIDGE ATIVA EM:', m.group(0))
        print('========================================\n')
        threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()
        break
